# Notebook Title: [Clear and Descriptive Title]

**Project**: Vision Transformer for Network Traffic Analysis  
**Topic**: [Specific topic covered]  
**Level**: Basic | Intermediate | Advanced  
**Duration**: [Estimated time] minutes  
**GPU Required**: Yes/No  

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
1. [First specific, measurable objective]
2. [Second specific, measurable objective]
3. [Third specific, measurable objective]

## 📋 Prerequisites

- **Knowledge**: [List required concepts]
- **Notebooks**: Complete [01_notebook_name.ipynb](01_notebook_name.ipynb) first
- **Setup**: Environment configured as per [quick_start_guide.md](../quick_start_guide.md)

## 📚 Table of Contents

1. [Introduction](#introduction)
2. [Setup and Imports](#setup)
3. [Data Preparation](#data-preparation)
4. [Main Content](#main-content)
5. [Experiments](#experiments)
6. [Exercises](#exercises)
7. [Summary](#summary)
8. [Further Reading](#further-reading)

## 1. Introduction <a id='introduction'></a>

[Provide context and motivation for this notebook's content. Explain why this topic is important for the overall project.]

### What You'll Build

[Describe the concrete outcome of this notebook]

### Key Concepts Preview

- **Concept 1**: Brief explanation
- **Concept 2**: Brief explanation
- **Concept 3**: Brief explanation

## 2. Setup and Imports <a id='setup'></a>

Let's start by importing necessary libraries and setting up our environment.

In [ ]:
# Standard library imports
import os
import sys
import time
import warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Machine learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Project specific imports
sys.path.append('../src')
# Uncomment when modules are available:
# from packet_encoder import PacketImageEncoder
# from vit_model import VisionTransformerClassifier
# from utils import set_seed, timer

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

# Display environment info
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

### Configuration Parameters

In [ ]:
# Model configuration
IMAGE_SIZE = 224          # Standard ViT input size
PATCH_SIZE = 16          # Size of each patch (16x16)
NUM_CLASSES = 2          # Binary classification (benign/malware)
EMBEDDING_DIM = 768      # Transformer embedding dimension

# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data configuration
MAX_PACKET_SIZE = 1500   # Standard MTU
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1

# Visualization configuration
FIGSIZE = (12, 8)
DPI = 100

# Display configuration summary
config_df = pd.DataFrame({
    'Parameter': ['Image Size', 'Patch Size', 'Batch Size', 'Learning Rate', 'Device'],
    'Value': [f"{IMAGE_SIZE}x{IMAGE_SIZE}", f"{PATCH_SIZE}x{PATCH_SIZE}", 
              BATCH_SIZE, LEARNING_RATE, str(DEVICE)]
})

display(HTML(config_df.to_html(index=False)))
print(f"\nTotal patches per image: {(IMAGE_SIZE // PATCH_SIZE) ** 2}")

### Helper Functions

In [ ]:
def setup_plot_style():
    """Set consistent plot styling across notebook."""
    plt.rcParams.update({
        'figure.figsize': FIGSIZE,
        'figure.dpi': DPI,
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'figure.titlesize': 18
    })

def print_memory_usage(stage: str = "Current"):
    """Print current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**2
        cached = torch.cuda.memory_reserved() / 1024**2
        print(f"💾 {stage} GPU memory: {allocated:.2f}MB allocated, {cached:.2f}MB cached")

# Apply styling
setup_plot_style()
print_memory_usage("Initial")

## 3. Data Preparation <a id='data-preparation'></a>

[Explain the data preparation steps specific to this notebook]

In [ ]:
# Example data loading code
# This is a placeholder - replace with actual data loading

def load_sample_data(num_samples: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    """Load sample packet data for demonstration."""
    # Generate synthetic packet data
    packet_lengths = np.random.randint(64, MAX_PACKET_SIZE, num_samples)
    packets = []
    labels = []
    
    for i, length in enumerate(packet_lengths):
        # Create synthetic packet
        if i < num_samples // 2:
            # Benign pattern
            packet = np.random.randint(0, 256, length, dtype=np.uint8)
            labels.append(0)
        else:
            # Malware pattern (simplified)
            packet = np.concatenate([
                np.array([0x4d, 0x5a]),  # PE header
                np.full(100, 0x90),      # NOP sled
                np.random.randint(0, 256, length - 102, dtype=np.uint8)
            ])[:length]
            labels.append(1)
        packets.append(packet)
    
    return packets, np.array(labels)

# Load sample data
print("Loading sample data...")
packets, labels = load_sample_data(1000)
print(f"✅ Loaded {len(packets)} packets")
print(f"   - Benign: {(labels == 0).sum()}")
print(f"   - Malware: {(labels == 1).sum()}")

### Data Exploration

In [ ]:
# Analyze packet characteristics
packet_lengths = [len(p) for p in packets]

# Create summary statistics
stats_df = pd.DataFrame({
    'Class': ['Benign', 'Malware'],
    'Count': [(labels == 0).sum(), (labels == 1).sum()],
    'Avg Length': [
        np.mean([len(p) for p, l in zip(packets, labels) if l == 0]),
        np.mean([len(p) for p, l in zip(packets, labels) if l == 1])
    ],
    'Std Length': [
        np.std([len(p) for p, l in zip(packets, labels) if l == 0]),
        np.std([len(p) for p, l in zip(packets, labels) if l == 1])
    ]
})

display(HTML(stats_df.round(2).to_html(index=False)))

# Visualize packet length distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
benign_lengths = [len(p) for p, l in zip(packets, labels) if l == 0]
malware_lengths = [len(p) for p, l in zip(packets, labels) if l == 1]

ax1.hist(benign_lengths, bins=30, alpha=0.7, label='Benign', color='#2E7D32')
ax1.hist(malware_lengths, bins=30, alpha=0.7, label='Malware', color='#D32F2F')
ax1.set_xlabel('Packet Length (bytes)')
ax1.set_ylabel('Count')
ax1.set_title('Packet Length Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot
data_to_plot = [benign_lengths, malware_lengths]
bp = ax2.boxplot(data_to_plot, labels=['Benign', 'Malware'], patch_artist=True)

colors = ['#2E7D32', '#D32F2F']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_ylabel('Packet Length (bytes)')
ax2.set_title('Packet Length by Class')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Packet Data Characteristics', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Main Content <a id='main-content'></a>

[This is where the main educational content goes. Structure it with clear subsections.]

### 4.1 [First Major Topic]

[Explanation of the concept before implementation]

In [ ]:
# Implementation example
# [Add your main implementation code here]

### 4.2 [Second Major Topic]

[Continue with additional topics as needed]

## 5. Experiments <a id='experiments'></a>

Let's experiment with different approaches and compare results.

In [ ]:
# Experiment setup
experiment_results = []

# Run experiments
# [Add experiment code here]

# Visualize results
# [Add visualization code here]

## 6. Exercises <a id='exercises'></a>

### 🏋️ Exercise 1: [Exercise Title]

**Objective**: [What the student will practice]

**Requirements**:
- Requirement 1
- Requirement 2

**Hint**: [Provide a helpful hint]

In [ ]:
# TODO: Implement your solution here
def exercise_solution():
    """
    Your implementation here.
    """
    pass

# Test your solution
# exercise_solution()

<details>
<summary>💡 Click to see solution</summary>

```python
def exercise_solution():
    # Solution implementation
    pass
```
</details>

### 🏋️ Exercise 2: [Exercise Title]

[Add more exercises as appropriate]

## 7. Summary <a id='summary'></a>

### 📝 What We Learned

1. **Key Concept 1**: [Summary of what was learned]
2. **Key Concept 2**: [Summary of what was learned]
3. **Key Concept 3**: [Summary of what was learned]

### 💻 Key Code Patterns

```python
# Most important pattern from this notebook
# example_code_here()
```

### ⚡ Performance Insights

- Insight 1: [Performance observation]
- Insight 2: [Performance observation]

### ⚠️ Common Pitfalls

- **Pitfall 1**: [Description and how to avoid]
- **Pitfall 2**: [Description and how to avoid]

## 8. Further Reading <a id='further-reading'></a>

### 📚 Papers
1. [Paper Title](link) - Brief description
2. [Paper Title](link) - Brief description

### 🔗 Related Notebooks
- Previous: [XX_previous_topic.ipynb](XX_previous_topic.ipynb)
- Next: [XX_next_topic.ipynb](XX_next_topic.ipynb)

### 🌐 External Resources
- [Resource 1](link) - Description
- [Resource 2](link) - Description

### 🚀 Next Steps
Ready to continue? Move on to [XX_next_notebook.ipynb](XX_next_notebook.ipynb) to learn about [next topic]!

---

## 🧹 Cleanup

In [ ]:
# Optional: Clean up resources
print_memory_usage("Final")

# Clear CUDA cache if using GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ CUDA cache cleared")

print("\n🎉 Notebook completed successfully!")